<a href="https://colab.research.google.com/github/smduarte/spbd-2526/blob/main/docs/labs/lab2/SPBD_Labs_mapreduce2_exercise_sol_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word Count Exercises

Count the number of occurrences of each word...

a) No sorting required, words and frequency can appear in any order;

b) Sorted by word, in increasing alphabetical order;

c) Sorted by frequency (the words with higher occurrence first).

In [ ]:
#@title Download the input file
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0
!pip install unidecode

In [ ]:
#@title Pure Python Reference Solution 1a)
import string
from unidecode import unidecode # used to remove accents and such...

def simple_word_count(file_path):
    word_counts = {}
    with open(file_path, 'r') as f:
        for line in f:
            # Remove accents and punctuation and convert to lowercase
            line = unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower()
            words = line.split()
            for word in words:
                word_counts[word] = word_counts.get(word, 0) + 1
    return word_counts

# Specify the path to the downloaded file
file_path = 'os_maias.txt'
word_counts = simple_word_count(file_path)

# Print the word counts
for word, count in word_counts.items():
    print(f'{word}: {count}')

In [ ]:
#@title Pure Python Reference Solution 1b)

file_path = 'os_maias.txt'
word_counts = simple_word_count(file_path)

for word, count in sorted(word_counts.items()):
    print(f'{word}: {count}')

In [ ]:
#@title Pure Python Reference Solution 1c)

file_path = 'os_maias.txt'
word_counts = simple_word_count(file_path)

for word, count in sorted(word_counts.items(), key=lambda item: item[1]):
    print(f'{word}: {count}')

# Python MapReduce Exercises

##1. MrJob MapReduce Word Frequency

Using the [MrJob](https://mrjob.readthedocs.io/en/latest/) library, create a map-reduce program that counts the number of occurrences of each word.

**a)** No sorting required, words and frequency can appear in any order;

**b)** Sorted by word, in increasing alphabetical order;

**c)** Sorted by frequency (the words with higher occurrence first).

**IMPORTANT**: Check the MrJob documentation to see how multi-step MapReduce jobs can
be implemented in the same Python class.



In [ ]:
#@title Download the Dataset and Install MrJob
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0
!pip install mrjob unidecode --quiet
!wget -q -O /etc/mrjob.conf https://raw.githubusercontent.com/smduarte/spbd-2526/main/docs/labs/lab1/mrjob.conf

In [ ]:
#@title Exercise 1a)
%%file word_freq_1a.py

import string
from unidecode import unidecode # used to remove accents and such...

from mrjob.job import MRJob, MRStep

class MRWordCountFrequency1a(MRJob):

  def mapper(self, _, line):
    line = unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower()
    words = line.split()
    for word in words:
      yield word, 1

  def combiner(self, word, occurrences ):
      yield word, sum( occurrences )

  def reducer(self, word, occurrences):
      yield word, sum( occurrences )

if __name__ == '__main__':
    MRWordCountFrequency1a.run()

In [ ]:
!rm -rf results
!python3 -m word_freq_1a -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/*

In [ ]:
#@title Exercise 1b)
%%file word_freq_1b.py

import string
from unidecode import unidecode # used to remove accents and such...

from mrjob.job import MRJob, MRStep

class MRWordCountFrequency1b(MRJob):

  def mapper(self, _, line):
    line = unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower()
    words = line.split()
    for word in words:
      yield word, 1

  def combiner(self, word, occurrences ):
      yield word, sum( occurrences )

  def reducer(self, word, occurrences):
      yield word, sum( occurrences )

if __name__ == '__main__':
    MRWordCountFrequency1b.run()

In [ ]:
!rm -rf results
!python3 -m word_freq_1b -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

In [ ]:
#@title Exercise 1c)
%%file word_freq_1c.py

import string
from unidecode import unidecode # used to remove accents and such...

from mrjob.job import MRJob, MRStep

MAX_FREQ = 1000000
class MRWordCountFrequency1c(MRJob):

  def mapper_frq(self, _, line):
    line = unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower()
    words = line.split()
    for word in words:
      yield word, 1

  def combiner_frq(self, word, occurrences ):
      yield word, sum( occurrences )

  def reducer_frq(self, word, occurrences):
      yield word, sum( occurrences )

  def mapper_sort(self, word, occurrences):
      yield "{:06}".format(MAX_FREQ-int(occurrences)), word

  def reducer_sort(self, occurrences, words):
    for word in words:
      yield word, MAX_FREQ - int(occurrences)

  def steps(self):
    return [ MRStep(mapper=self.mapper_frq, combiner=self.combiner_frq, reducer=self.reducer_frq),
      MRStep(mapper=self.mapper_sort, reducer=self.reducer_sort)
    ]

if __name__ == '__main__':
    MRWordCountFrequency1c.run()


In [ ]:
!rm -rf results
!python3 -m word_freq_1c -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

In [ ]:
#@title Exercise 1c v2)
%%file word_freq_1c_v2.py

import string
from unidecode import unidecode # used to remove accents and such...

from mrjob.job import MRJob, MRStep

MAX_FREQ = 1000000
class MRWordCountFrequency1cV2(MRJob):

  def mapper_frq(self, _, line):
    line = unidecode(line).translate(str.maketrans('', '', string.punctuation+'«»')).lower()
    words = line.split()
    for word in words:
      yield word, 1

  def combiner_frq(self, word, occurrences):
      yield word, sum(occurrences)

  def reducer_frq(self, word, occurrences):
      yield "{:06}".format(MAX_FREQ - sum( occurrences )), word

  def reducer_sort(self, occurrences, words):
    for word in words:
      yield word, MAX_FREQ - int(occurrences)

  def steps(self):
    return [ MRStep(mapper=self.mapper_frq, combiner=self.combiner_frq, reducer=self.reducer_frq),
      MRStep(reducer=self.reducer_sort)]

if __name__ == '__main__':
    MRWordCountFrequency1cV2.run()


In [ ]:
!rm -rf results
!python3 -m word_freq_1c_v2 -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10